In [ ]:
#Load libraries
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, roc_auc_score, f1_score, average_precision_score,
    confusion_matrix, precision_score, recall_score, balanced_accuracy_score
)


In [2]:
#Load PRS score and labels
PRS_df = pd.read_csv("../../results/PRS/4_PRS_calculation/score_final_w_labels.txt", sep="\t")

In [4]:
# Map cancer types to binary labels
category_mapping = {'Control': 0, 'Colorectal': 1}
PRS_df = pd.read_csv("../../results/PRS/4_PRS_calculation/score_final_w_labels.txt", sep="\t")
PRS_df['Cancer_Map'] = PRS_df['tipo_cancer'].map(category_mapping)
PRS_df.head()

,IID,NMISS_ALLELE_CT,ALLELE_DOSAGE_SUM,SCORE1_AVG,SCORE1_AVG_scale,tipo_cancer,Cancer_Map
0,A00002,398,184,0.029502,-0.444913,Control,0
1,A00005,398,187,0.028999,-0.812958,Control,0
2,A00011,398,188,0.029138,-0.711252,Control,0
3,A00012,398,176,0.028448,-1.216563,Control,0
4,A00014,398,186,0.029614,-0.362963,Control,0


In [5]:
# Select X and y for logistic regression
X = PRS_df[['SCORE1_AVG_scale']]
y = PRS_df[['Cancer_Map']]

In [7]:
# Fit logistic regression model
model = LogisticRegression()
model.fit(X, y) 

c:\Users\Zaid\miniconda3\envs\ML\lib\site-packages\sklearn\utils\validation.py:1406: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [8]:
# Print model coefficients and intercept
print("Coefficients:", model.coef_)
print("Intercept:", model.intercept_)

Coefficients: [[0.47400707]]
Intercept: [-0.87908165]


In [9]:
# Calculate odds ratio
np.exp(model.coef_[0])

array([1.60641834])

In [15]:
# Predict class labels
y_pred = model.predict(X)

# Probabilities
y_pred_proba = model.predict_proba(X)[:, 1]

# Accuracy
acc = round(accuracy_score(y, y_pred), 4)
print("Accuracy:", acc)

# ROC AUC
auc = round(roc_auc_score(y, y_pred_proba), 4)
print("ROC AUC:", auc)

# F1 Score
f1 = round(f1_score(y, y_pred), 4)
print("F1 Score:", f1)

# PR AUC
pr_auc = round(average_precision_score(y, y_pred_proba), 4)
print("PR AUC:", pr_auc)

# Confusion Matrix
tn, fp, fn, tp = confusion_matrix(y, y_pred).ravel()

# Sensitivity (Recall)
sensitivity = round(tp / (tp + fn), 4)
print("Sensitivity:", sensitivity)

# Specificity
specificity = round(tn / (tn + fp), 4)
print("Specificity:", specificity)

# PPV (Precision)
ppv = round(tp / (tp + fp), 4)
print("PPV:", ppv)

# NPV
npv = round(tn / (tn + fn), 4)
print("NPV:", npv)

Accuracy: 0.7025
ROC AUC: 0.6259
F1 Score: 0.1107
PR AUC: 0.4175
Sensitivity: 0.0613
Specificity: 0.9801
PPV: 0.5714
NPV: 0.7069


Logistic Regression for Genotype Analysis

In [16]:
#Load data
Effect_allele_count_df = pd.read_excel("../../results/PRS/4_PRS_calculation/effect_allele_counts.xlsx")

In [17]:
#Drop unnecessary columns
Effect_allele_count_df = Effect_allele_count_df.drop(columns=['PAT','MAT','SEX','PHENOTYPE','FID'])

In [18]:
#Merge with PRS_df to get cancer labels
Effect_allele_count_df = Effect_allele_count_df.merge(PRS_df[['IID','Cancer_Map']], left_on='IID', right_on='IID')

In [19]:
#Split into train and test sets with stratification with 80-20 split
train_df, test_df = train_test_split(
    Effect_allele_count_df,
    test_size=0.2,
    stratify=Effect_allele_count_df['Cancer_Map'],
    random_state=42
)

In [20]:
#Select X and y for logistic regression
X_train = train_df.drop(columns=['Cancer_Map', 'IID'])
y_train = train_df['Cancer_Map']

X_test = test_df.drop(columns=['Cancer_Map', 'IID'])
y_test = test_df['Cancer_Map']

In [22]:
#Fit logistic regression model
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [28]:
# Predict class labels
y_pred = model.predict(X_test)

# Probabilities
y_pred_proba = model.predict_proba(X_test)[:, 1]

# Accuracy
acc = round(accuracy_score(y_test, y_pred), 4)
print("Accuracy:", acc)

# ROC AUC
auc = round(roc_auc_score(y_test, y_pred_proba), 4)
print("ROC AUC:", auc)

# F1 Score
f1 = round(f1_score(y_test, y_pred), 4)
print("F1 Score:", f1)

# PR AUC
pr_auc = round(average_precision_score(y_test, y_pred_proba), 4)
print("PR AUC:", pr_auc)

# Confusion Matrix
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

# Sensitivity (Recall)
sensitivity = round(tp / (tp + fn), 4)
print("Sensitivity:", sensitivity)

# Specificity
specificity = round(tn / (tn + fp), 4)
print("Specificity:", specificity)

# PPV (Precision)
ppv = round(tp / (tp + fp), 4)
print("PPV:", ppv)

# NPV
npv = round(tn / (tn + fn), 4)
print("NPV:", npv)

Accuracy: 0.6931
ROC AUC: 0.5705
F1 Score: 0.2881
PR AUC: 0.3722
Sensitivity: 0.2056
Specificity: 0.904
PPV: 0.4811
NPV: 0.7245


LASSO Regression

In [ ]:
#Fit Lasso logistic regression model
lasso_model = LogisticRegression(
    penalty='l1',
    solver='liblinear',   # supports L1
    max_iter=1000
)

lasso_model.fit(X_train, y_train)

,penalty,'l1'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'liblinear'
,max_iter,1000
,multi_class,'deprecated'


In [ ]:
#Identify which predictors were removed (coefficients set to zero) and which were selected (non-zero coefficients)
coef_df = pd.DataFrame({
    "feature": X_train.columns,
    "coefficient": lasso_model.coef_[0]
})
removed_predictors = coef_df[coef_df["coefficient"] == 0]
selected_predictors = coef_df[coef_df["coefficient"] != 0]

print("Removed predictors:")
print(removed_predictors)

print("Selected predictors:")
print(selected_predictors)

Removed predictors:
               feature  coefficient
19   10:81048611_C_T_C          0.0
28   11:69938433_A_G_A          0.0
30   11:74409077_T_C_T          0.0
52   13:27543193_C_T_C          0.0
54   13:73649152_A_G_A          0.0
67   14:93014929_G_A_G          0.0
93   19:49218602_C_T_C          0.0
97   2:136407479_A_G_A          0.0
125  22:46364191_C_T_C          0.0
145    5:1240998_T_C_T          0.0
169   6:55577214_T_C_T          0.0
Selected predictors:
               feature  coefficient
0     1:22503282_G_C_G     0.090391
1     1:22710877_G_C_G    -0.001885
2     1:38461319_C_A_C     0.032501
3     1:55247852_A_G_A    -0.053197
4     1:62673037_T_C_T     0.022670
..                 ...          ...
194  9:101679752_T_G_T     0.106792
195  9:110373819_C_T_C     0.018855
196  9:113654654_G_C_G     0.011782
197  9:136682468_C_T_C     0.105024
198  9:136925663_G_T_G    -0.075743

[188 rows x 2 columns]


In [29]:
# Predict class labels
y_pred = model.predict(X_test)

# Probabilities
y_pred_proba = model.predict_proba(X_test)[:, 1]

# Accuracy
acc = round(accuracy_score(y_test, y_pred), 4)
print("Accuracy:", acc)

# ROC AUC
auc = round(roc_auc_score(y_test, y_pred_proba), 4)
print("ROC AUC:", auc)

# F1 Score
f1 = round(f1_score(y_test, y_pred), 4)
print("F1 Score:", f1)

# PR AUC
pr_auc = round(average_precision_score(y_test, y_pred_proba), 4)
print("PR AUC:", pr_auc)

# Confusion Matrix
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

# Sensitivity (Recall)
sensitivity = round(tp / (tp + fn), 4)
print("Sensitivity:", sensitivity)

# Specificity
specificity = round(tn / (tn + fp), 4)
print("Specificity:", specificity)

# PPV (Precision)
ppv = round(tp / (tp + fp), 4)
print("PPV:", ppv)

# NPV
npv = round(tn / (tn + fn), 4)
print("NPV:", npv)

Accuracy: 0.6931
ROC AUC: 0.5705
F1 Score: 0.2881
PR AUC: 0.3722
Sensitivity: 0.2056
Specificity: 0.904
PPV: 0.4811
NPV: 0.7245
